# End to end: one sentence, every stage, real numbers

One real prompt, traced through the entire pipeline in one place — tokenizer → embeddings → positional encoding → every transformer block's real attention → logits → softmax → sampling. Every number below comes from your actual trained checkpoint, not an illustration.

The prompt is chosen deliberately: **"ROMEO: the king said the king"** — the word `king` tokenizes to the *exact same* token id both times it appears. That lets us ask a real question: when the model reaches the second `king`, does any attention head look back at what immediately FOLLOWED the first `king`, to predict what should come next? That behaviour is called an **induction head**, and it's the actual mechanism behind the sudden training "breakthrough" you saw earlier — this is our chance to check, on your own model, whether it actually formed one.

In [1]:
import torch
import torch.nn.functional as F
from model.gpt import GPT
from model.sample import sample
from tokenizer.bpe import BPETokenizer

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

tok = BPETokenizer()
tok.load("tokenizer/vocab.json")

ckpt = torch.load("checkpoints/gpt_shakespeare.pt", map_location=device)
model = GPT(**ckpt["config"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()   # IMPORTANT: turns dropout off, so this trace is deterministic
               # and matches a real forward() call exactly, number for number

print(f"device: {device}  |  iteration {ckpt['iter']}  |  {model.num_params():,} params")

device: mps  |  iteration 6600  |  922,880 params


## Stage 1 — Tokenization

In [2]:
prompt = "ROMEO: the king said the king"
ids = tok.encode(prompt)

print(f"{prompt!r}")
print(f"-> {len(prompt)} characters -> {len(ids)} tokens ({len(prompt)/len(ids):.2f} chars/token)\n")
for pos, i in enumerate(ids):
    print(f"  pos {pos}: id {i:4d}  {tok.decode([i])!r}")

king_positions = [pos for pos, i in enumerate(ids) if i == 511]
print(f"\n'king' (token id 511) appears at positions: {king_positions}")
assert len(king_positions) == 2, "expected exactly two identical 'king' tokens"
FIRST_KING, SECOND_KING = king_positions

'ROMEO: the king said the king'
-> 29 characters -> 10 tokens (2.90 chars/token)

  pos 0: id  923  'ROME'
  pos 1: id   79  'O'
  pos 2: id   58  ':'
  pos 3: id  304  ' the '
  pos 4: id  511  'king'
  pos 5: id  282  ' s'
  pos 6: id   97  'a'
  pos 7: id  614  'id '
  pos 8: id  295  'the '
  pos 9: id  511  'king'

'king' (token id 511) appears at positions: [4, 9]


## Stage 2 — Embeddings (real ones now, not milestone 2's random noise)

In [3]:
ids_t = torch.tensor([ids], device=device)
with torch.no_grad():
    tok_emb = model.token_emb(ids_t)   # (1, seq_len, n_embd)

print(f"embedded shape: {tuple(tok_emb.shape)}")
print(f"first 6 dims of 'king' at position {FIRST_KING}:  {tok_emb[0, FIRST_KING, :6].tolist()}")
print(f"first 6 dims of 'king' at position {SECOND_KING}: {tok_emb[0, SECOND_KING, :6].tolist()}")
same = torch.equal(tok_emb[0, FIRST_KING], tok_emb[0, SECOND_KING])
print(f"identical before positional encoding: {same}  (it's the same lookup row both times)")

embedded shape: (1, 10, 128)
first 6 dims of 'king' at position 4:  [-0.046650175005197525, 0.021479249000549316, 0.0837060734629631, -0.10797037929296494, -0.014150233939290047, 0.04682902246713638]
first 6 dims of 'king' at position 9: [-0.046650175005197525, 0.021479249000549316, 0.0837060734629631, -0.10797037929296494, -0.014150233939290047, 0.04682902246713638]
identical before positional encoding: True  (it's the same lookup row both times)


## Stage 3 — Positional encoding

In [4]:
with torch.no_grad():
    x = model.pos_enc.add_to(tok_emb)   # (1, seq_len, n_embd) -- the actual input to block 0

same_after = torch.equal(x[0, FIRST_KING], x[0, SECOND_KING])
diff = (x[0, FIRST_KING] - x[0, SECOND_KING]).abs().max().item()
print(f"still identical after adding position info: {same_after}")
print(f"max difference between the two 'king' vectors now: {diff:.4f}")
print("-> same word, different position, genuinely different vector -- position 3's job, done.")

still identical after adding position info: False
max difference between the two 'king' vectors now: 1.8834
-> same word, different position, genuinely different vector -- position 3's job, done.


## Stage 4 — Through every transformer block, and the induction-head check

For each of the 4 blocks: track how much the hidden state at the second `king` (position 9) has moved (its representation is being built up, layer by layer), and check every head's real attention weights at that position. Specifically: how much weight does the query at the SECOND `king` place on position `FIRST_KING + 1` — the token that immediately followed `king` the first time? If any head does this well above baseline, that's an induction head.

In [5]:
target_pos = FIRST_KING + 1   # the token right after 'king' the FIRST time -- what an
                              # induction head should look back at, from the second 'king'
print(f"looking for attention from position {SECOND_KING} ('king', 2nd time) "
      f"back to position {target_pos} ({tok.decode([ids[target_pos]])!r}, what followed 'king' last time)\n")

best = {"score": -1}
with torch.no_grad():
    for layer_idx, block in enumerate(model.blocks):
        prev_norm = x[0].norm(dim=-1).mean().item()

        normed = block.ln1(x)
        for head_idx, head in enumerate(block.attn.heads):
            _, weights = head(normed, return_weights=True)   # (1, T, T)
            row = weights[0, SECOND_KING]                    # this head's attention FROM the 2nd 'king'
            w_target = row[target_pos].item()                # weight on "what followed king last time"
            w_prev   = row[SECOND_KING - 1].item()            # weight on the IMMEDIATELY preceding token (recency baseline)
            print(f"layer {layer_idx} head {head_idx}: weight on target={w_target:.3f}  "
                  f"vs. weight on prev-token={w_prev:.3f}")
            if w_target > best["score"]:
                best = {"score": w_target, "layer": layer_idx, "head": head_idx, "row": row.tolist()}

        # advance x through the real block (attention + feed-forward + both residuals)
        x = block(x)
        new_norm = x[0].norm(dim=-1).mean().item()
        print(f"  -- after block {layer_idx}: mean hidden-state norm {prev_norm:.2f} -> {new_norm:.2f}\n")

print(f"strongest candidate: layer {best['layer']} head {best['head']}, "
      f"weight {best['score']:.3f} on the induction target")
print(f"full attention row from position {SECOND_KING}:")
for pos, w in enumerate(best["row"]):
    marker = "  <- induction target" if pos == target_pos else ("  <- 'king' itself" if pos == SECOND_KING else "")
    print(f"  pos {pos} ({tok.decode([ids[pos]])!r}): {w:.3f}{marker}")

looking for attention from position 9 ('king', 2nd time) back to position 5 (' s', what followed 'king' last time)

layer 0 head 0: weight on target=0.082  vs. weight on prev-token=0.039
layer 0 head 1: weight on target=0.076  vs. weight on prev-token=0.019
layer 0 head 2: weight on target=0.075  vs. weight on prev-token=0.036
layer 0 head 3: weight on target=0.101  vs. weight on prev-token=0.023
  -- after block 0: mean hidden-state norm 8.04 -> 8.47

layer 1 head 0: weight on target=0.038  vs. weight on prev-token=0.211
layer 1 head 1: weight on target=0.065  vs. weight on prev-token=0.140
layer 1 head 2: weight on target=0.068  vs. weight on prev-token=0.107
layer 1 head 3: weight on target=0.052  vs. weight on prev-token=0.111
  -- after block 1: mean hidden-state norm 8.47 -> 9.25

layer 2 head 0: weight on target=0.002  vs. weight on prev-token=0.295
layer 2 head 1: weight on target=0.040  vs. weight on prev-token=0.162
layer 2 head 2: weight on target=0.022  vs. weight on prev-t

## Stage 5 — Final layer norm + output head → logits

In [6]:
with torch.no_grad():
    x_final = model.ln_f(x)
    logits = model.lm_head(x_final)[0, -1]   # last position's logits -- what comes after our prompt

print(f"logits shape: {tuple(logits.shape)}  (one raw score per vocab token)")
print(f"min={logits.min().item():.2f}  max={logits.max().item():.2f}  mean={logits.mean().item():.2f}")
print(f"argmax token (most likely, unfiltered): {tok.decode([logits.argmax().item()])!r}")

logits shape: (1024,)  (one raw score per vocab token)
min=-8.45  max=7.89  mean=-2.07
argmax token (most likely, unfiltered): ',\n'


## Stage 6 — Softmax → real probabilities

In [7]:
probs = F.softmax(logits, dim=-1)
top_probs, top_ids = torch.topk(probs, 10)
print(f"real top-10 next-token probabilities after {prompt!r}:")
for p, i in zip(top_probs.tolist(), top_ids.tolist()):
    print(f"  {p*100:5.1f}%  {tok.decode([i])!r}")

real top-10 next-token probabilities after 'ROMEO: the king said the king':
    9.4%  ',\n'
    7.6%  '\n'
    5.4%  '.\n\n'
    5.3%  '.\n'
    5.3%  "'s "
    4.6%  's'
    3.8%  ':\n'
    3.2%  ', '
    3.0%  ';\n'
    3.0%  '?\n\n'


## Stage 7 — Sampling → the actual chosen token, and the rest of the sentence

In [8]:
next_id = sample(logits, temperature=0.8, top_k=40).item()
print(f"sampled: {tok.decode([next_id])!r}")
print(f"(what followed 'king' the FIRST time was {tok.decode([ids[target_pos]])!r} — worth comparing)\n")

@torch.no_grad()
def generate(prompt, max_new_tokens=40, temperature=0.8, top_k=40):
    out_ids = tok.encode(prompt)
    for _ in range(max_new_tokens):
        xg = torch.tensor([out_ids[-model.block_size:]], device=device)
        lg = model(xg)
        out_ids.append(sample(lg[0, -1], temperature=temperature, top_k=top_k).item())
    return tok.decode(out_ids)

print("full continuation:")
print(generate(prompt))

sampled: ' '
(what followed 'king' the FIRST time was ' s' — worth comparing)

full continuation:
ROMEO: the king said the king;
But by your howy and the feed, if I not becaco!
And be the a dake him are hold a more plin:
'Tis 
